

Синтез відповідей трьох LLM через Groq API:
- **Llama 3.3 70B** (`llama-3.3-70b-versatile`)
- **Llama 3.1 8B** (`llama-3.1-8b-instant`)
- **Llama 4 Scout** (`meta-llama/llama-4-scout-17b-16e-instruct`)



In [1]:
!pip install groq -q


[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os, getpass, time, json
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from groq import Groq

GROQ_API_KEY = os.environ.get("GROQ_API_KEY") or getpass.getpass("Groq API Key: ")
client = Groq(api_key=GROQ_API_KEY)

MODEL_A = "llama-3.3-70b-versatile"
MODEL_B = "llama-3.1-8b-instant"
MODEL_C = "meta-llama/llama-4-scout-17b-16e-instruct"

SYSTEM = (
    "Ти — AI-асистент водія концепт-кара. "
    "Відповідай українською, чітко і практично, 2-3 речення, без форматування."
)

Groq API Key:  ········


In [3]:
@dataclass
class Situation:
    name: str
    prompt_a: str
    prompt_b: str
    prompt_c: str

SITUATIONS = [
    Situation(
        name="Мертва зона дзеркал",
        prompt_a="Автомобіль потрапив у мертву зону дзеркал іншого авто. Що робити водієві? 2-3 речення.",
        prompt_b="Як безпечно виїхати з мертвої зони дзеркал сусіднього автомобіля? 2-3 речення.",
        prompt_c="Водій опинився у сліпій зоні поруч їдучого авто. Які дії? 2-3 речення.",
    ),
    Situation(
        name="Проблеми з тиском шин",
        prompt_a="TPMS попередила про низький тиск під час руху. Перші дії водія? 2-3 речення.",
        prompt_b="Раптове спускання шини на трасі — що робити? 2-3 речення.",
        prompt_c="Датчик тиску шин сигналізує про проблему. Як діяти? 2-3 речення.",
    ),
    Situation(
        name="Очищення лобового скла",
        prompt_a="Відмовили склоочисники у сильний дощ. Що робити? 2-3 речення.",
        prompt_b="Лобове скло обледеніло або запотіло зсередини. Дії водія? 2-3 речення.",
        prompt_c="Погана видимість — дворники не працюють. Що робити? 2-3 речення.",
    ),
    Situation(
        name="Дорожньо-транспортна пригода",
        prompt_a="Щойно сталася ДТП. Які перші дії водія? 2-3 речення.",
        prompt_b="Після аварії водій у шоці. Що зробити в першу чергу? 2-3 речення.",
        prompt_c="ДТП сталося. Безпека, виклик допомоги, документування. 2-3 речення.",
    ),
    Situation(
        name="Відмова двигуна",
        prompt_a="Двигун раптово заглух на трасі. Дії водія наступні 30 секунд? 2-3 речення.",
        prompt_b="Відмовив двигун — кермо і гальма важчають. Як діяти? 2-3 речення.",
        prompt_c="Двигун зупинився під час їзди. Як безпечно зупинитися? 2-3 речення.",
    ),
    Situation(
        name="Ожеледиця та занос",
        prompt_a="Автомобіль пішов у занос на ожеледиці. Як повернути контроль? 2-3 речення.",
        prompt_b="Льодяна дорога — машина не реагує на кермо. Дії водія? 2-3 речення.",
        prompt_c="Задня вісь пішла в занос на льоду. Що робити і чого уникати? 2-3 речення.",
    ),
]

In [4]:
def ask(model: str, prompt: str) -> str:
    try:
        r = client.chat.completions.create(
            model=model, max_tokens=300, temperature=0.7,
            messages=[{"role": "system", "content": SYSTEM},
                      {"role": "user",   "content": prompt}]
        )
        return r.choices[0].message.content.strip()
    except Exception as e:
        return f"Помилка: {e}"


def run(sit: Situation) -> dict:
    tasks = {
        "Llama 3.3 70B": (MODEL_A, sit.prompt_a),
        "Llama 3.1 8B":  (MODEL_B, sit.prompt_b),
        "Llama 4 Scout": (MODEL_C, sit.prompt_c),
    }
    responses = {}
    with ThreadPoolExecutor(max_workers=3) as ex:
        futures = {ex.submit(ask, m, p): name for name, (m, p) in tasks.items()}
        for f in as_completed(futures):
            responses[futures[f]] = f.result()

    synth_prompt = (
        f"Ситуація: '{sit.name}'.\n"
        f"Llama 3.3 70B: {responses['Llama 3.3 70B']}\n"
        f"Llama 3.1 8B: {responses['Llama 3.1 8B']}\n"
        f"Llama 4 Scout: {responses['Llama 4 Scout']}\n\n"
        f"Синтезуй 4 чіткі кроки для водія. Розділяй символом '|'. Без нумерації."
    )
    raw = ask(MODEL_A, synth_prompt)
    steps = [s.strip() for s in raw.split("|") if s.strip()]
    return {"responses": responses, "synthesis": steps}


def print_result(sit: Situation, result: dict):
    print(f"\n{'='*60}")
    print(f"  {sit.name}")
    print(f"{'='*60}")
    for model, text in result["responses"].items():
        print(f"\n[{model}]\n{text}")
    print(f"\n--- Синтез ---")
    for i, step in enumerate(result["synthesis"], 1):
        print(f"{i}. {step}")


def automind(text: str):
    """Довільна ситуація."""
    sit = Situation(
        name=text,
        prompt_a=f"Що робити водієві: {text}? 2-3 речення.",
        prompt_b=f"Яких дій вимагає ситуація '{text}'? 2-3 речення.",
        prompt_c=f"Порада водієві: {text}. 2-3 речення.",
    )
    print_result(sit, run(sit))

## Тест

In [5]:
sit = SITUATIONS[0]
print_result(sit, run(sit))


  Мертва зона дзеркал

[Llama 4 Scout]
Якщо ви опинились у сліпій зоні іншого авто, змініть смугу руху або швидкість, щоб бути поміченим. Якщо зміна смуги неможлива, збільште швидкість, щоб привернути увагу водія іншого авто. У разі небезпеки подайте звуковий сигнал.

[Llama 3.1 8B]
Виїхати з мертвої зони треба дуже повільно, не роблячи різких рухів, щоб уникнути різкого збільшення швидкості або несподіваних маневрів. Змініть швидкість відповідно до дорожніх умов та стану руху навколо автомобіля, особливо якщо попереду ви побачили вільну ділянку дороги. Дуже важливо бути уважним щодо дій інших учасників руху та бути готовим до можливої зміни ситуації.

[Llama 3.3 70B]
Якщо автомобіль потрапив у мертву зону дзеркал іншого авто, водієві слід бути надзвичайно обережним і намагатися якнайшвидше повернутися у видиму зону. Для цього можна збільшити швидкість або знижувати швидкість, щоб вийти з мертвої зони. Крім того, водієві варто користуватися зовнішніми дзеркалами та оглядаючи сідло сво

## Усі ситуації

In [6]:
all_results = []
for sit in SITUATIONS:
    result = run(sit)
    print_result(sit, result)
    all_results.append({"situation": sit.name, **result})
    time.sleep(0.5)


  Мертва зона дзеркал

[Llama 3.1 8B]
Під час виїзду з мертвої зони сусіднього автомобіля слід уважно спостерігати за ситуацією на дорозі та дзеркалах, щоб уникнути можливого відчуження або зіткнення. Нажмите кнопку розсвітлення, щоб збільшити видимість, та трохи повільно почніть рух, спостерігаючи за реакцією водія сусіднього автомобіля. Якщо він починає рух, продовжуйте рух вашого автомобіля, повільно набираючи швидкість згідно з ситуацією.

[Llama 3.3 70B]
Якщо ваш автомобіль потрапив у мертву зону дзеркал іншого авто, вам потрібно бути особливо уважним і спостерігати за дорогою та іншими учасниками руху. Для цього можна використовувати зовнішні дзеркала та оглядове дзеркало вашого автомобіля, а також регулярно перевіряти мертві зони. Крім того, вам варто бути готовим до можливих маневрів іншого водія та бути готовим до швидкої реакції.

[Llama 4 Scout]
Якщо ви опинились у сліпій зоні іншого автомобіля, змініть смугу руху або швидкість, щоб бути поміченим. Якщо зміна смуги неможлив

## Довільна ситуація

In [7]:
automind("засліпили фари зустрічного авто вночі")


  засліпили фари зустрічного авто вночі

[Llama 3.1 8B]
У випадку, коли фари зустрічного авто засліплюють, необхідно зменшити швидкість та віддалитися від нього на мінімальну необхідну відстань. Якщо це можливо, спробуйте змінити напрям руху або зупинитися. Також не забувайте про використання світлодіодного відбивача або відбивача світла на капоті, щоб зменшити відбиття світла.

[Llama 3.3 70B]
Якщо вас засліпили фари зустрічного авто вночі, то необхідно відвести погляд від фар і продовжити рух, орієнтуючись на правий край дороги або на лінію руху. Також можна тимчасово зменшити швидкість, щоб зберегти контроль над автомобілем. Якщо фари продовжують засліплювати, можна користуватися лівим зеркалом заднього виду, щоб зберегти орієнтацію на дорозі.

[Llama 4 Scout]
Не змінюйте різко напрямок руху, щоб не втратити контроль над автомобілем. Зменшіть швидкість і продовжуйте рух, дотримуючись правої сторони дороги. Увімкніть ближнє світло фар, щоб зменшити сліпучий ефект для інших водіїв.

